<a href="https://colab.research.google.com/github/jriveramerla/JRM-pySpark001/blob/master/JRM_Distributed_Processing_Challenges_Handling_Data_Skew_in_DF_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession

spark= SparkSession.builder.appName('Practice').getOrCreate()

sc = spark.sparkContext

# Loading DataSkew
To understand skew, we create random data where keys are uniformly distributed

In [4]:
import numpy as np
import pandas as pd
import random


# sale dataset:
# table 1: OrderID, Qty, Sales, Discount (yes=1, no=0)
# table 2: ProductID, OrderID, Product, Price

########### Table 1 ###########

key_1 = [101] * 100
key_2 = [201] * 7000000
key_3 = [301] * 500
key_4 = [401] * 10000
OrderID = key_1 + key_2 + key_3 + key_4
random.shuffle(OrderID)

Qty_1 = list(np.random.randint(low = 1, high = 100, size = len(key_1)))
Qty_2 = list(np.random.randint(low = 1, high = 200, size = len(key_2)))
Qty_3 = list(np.random.randint(low = 1, high = 1000, size = len(key_3)))
Qty_4 = list(np.random.randint(low = 1, high = 50, size = len(key_4)))
Qty = Qty_1 + Qty_2 + Qty_3 + Qty_4

Sales_1 = list(np.random.randint(low = 10, high = 100, size = len(key_1)))
Sales_2 = list(np.random.randint(low = 50, high = 3400, size = len(key_2)))
Sales_3 = list(np.random.randint(low = 12, high = 2000, size = len(key_3)))
Sales_4 = list(np.random.randint(low = 40, high = 1000, size = len(key_4)))
Sales = Sales_1 + Sales_2 + Sales_3 + Sales_4

Discount = list(np.random.randint(low = 0, high = 2, size = len(OrderID)))
data1 = list(zip(OrderID,Qty,Sales,Discount))

# Create the Pandas DF
data_skew = pd.DataFrame(data1, columns=['OrderID','Qty','Sales','Discount'])


########### Table 2 ###########
data2 = [[1, 101, 'pencil', 4.99],
         [2, 101, 'book', 9.5],
         [3, 101, 'scissors', 14],
         [4, 301, 'glue', 7],
         [5, 201, 'marker', 8.49],
         [6, 301, 'label', 2],
         [7, 201, 'calculator', 3.99],
         [8, 501, 'eraser', 1.55],
        ]

data_small = pd.DataFrame(data2, columns=['ProductID', 'OrderID', 'Product', 'Price'])

In [5]:
# Create PySpark DF from Pandas

# Optimize conversion between PySpark and Pandas DF: Enable arrow-based columnar data transfers
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

df_skew = spark.createDataFrame(data_skew)
df_skew.printSchema()
df_skew.show()
df_skew.rdd.getNumPartitions()

root
 |-- OrderID: long (nullable = true)
 |-- Qty: long (nullable = true)
 |-- Sales: long (nullable = true)
 |-- Discount: long (nullable = true)

+-------+---+-----+--------+
|OrderID|Qty|Sales|Discount|
+-------+---+-----+--------+
|    201| 67|   14|       0|
|    201| 22|   41|       1|
|    201| 41|   65|       1|
|    201| 32|   34|       0|
|    201| 43|   62|       0|
|    201| 37|   82|       1|
|    201| 74|   99|       0|
|    201| 58|   67|       1|
|    201| 88|   69|       1|
|    201| 49|   58|       0|
|    201| 60|   27|       1|
|    201| 38|   34|       1|
|    201| 87|   90|       1|
|    201| 43|   39|       0|
|    201| 81|   69|       1|
|    201| 41|   63|       1|
|    201| 54|   30|       1|
|    201| 16|   53|       1|
|    201| 83|   50|       1|
|    201| 86|   52|       1|
+-------+---+-----+--------+
only showing top 20 rows


702

In [6]:
df_small = spark.createDataFrame(data_small)
df_small.printSchema()
df_small.show()
df_small.rdd.getNumPartitions()

root
 |-- ProductID: long (nullable = true)
 |-- OrderID: long (nullable = true)
 |-- Product: string (nullable = true)
 |-- Price: double (nullable = true)

+---------+-------+----------+-----+
|ProductID|OrderID|   Product|Price|
+---------+-------+----------+-----+
|        1|    101|    pencil| 4.99|
|        2|    101|      book|  9.5|
|        3|    101|  scissors| 14.0|
|        4|    301|      glue|  7.0|
|        5|    201|    marker| 8.49|
|        6|    301|     label|  2.0|
|        7|    201|calculator| 3.99|
|        8|    501|    eraser| 1.55|
+---------+-------+----------+-----+



2

# (2) Run a shuffle Join() with small sized data¶


In [7]:
joined_df = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

joined_df.show(30)

print(joined_df.count())

+-------+---+-----+--------+---------+-------+----------+-----+
|OrderID|Qty|Sales|Discount|ProductID|OrderID|   Product|Price|
+-------+---+-----+--------+---------+-------+----------+-----+
|    201| 67|   14|       0|        7|    201|calculator| 3.99|
|    201| 67|   14|       0|        5|    201|    marker| 8.49|
|    201| 22|   41|       1|        7|    201|calculator| 3.99|
|    201| 22|   41|       1|        5|    201|    marker| 8.49|
|    201| 41|   65|       1|        7|    201|calculator| 3.99|
|    201| 41|   65|       1|        5|    201|    marker| 8.49|
|    201| 32|   34|       0|        7|    201|calculator| 3.99|
|    201| 32|   34|       0|        5|    201|    marker| 8.49|
|    201| 43|   62|       0|        7|    201|calculator| 3.99|
|    201| 43|   62|       0|        5|    201|    marker| 8.49|
|    201| 37|   82|       1|        7|    201|calculator| 3.99|
|    201| 37|   82|       1|        5|    201|    marker| 8.49|
|    201| 74|   99|       0|        7|  

In [ ]:
# DF increases the parition number to 200 automatically when spark performs data shuffling (join, aggregation)
print(joined_df.rdd.getNumPartitions())
joined_df.rdd.glom().collect()

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", 8)

# run join()
joined_df = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")
joined_df.show(30)



In [ ]:
print(joined_df.rdd.getNumPartitions())
joined_df.rdd.glom().collect()

In [ ]:
# perform a join() and descriptive statistics on a skewed data
from pyspark.sql.functions import *


groups = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

## Mitigate data skewness: SALTING¶


In [ ]:
from pyspark.sql.functions import *

# add  random value [0 2]
df_skew_salting = df_skew.withColumn("_salt_", round(rand() * 2))
df_small_salting = df_small.withColumn("_salt_", round(rand() * 2))

df_skew_salting.show()
df_small_salting.show()

In [ ]:
# repartition using _salt_:
df_skew_salting = df_skew_salting.repartition(3, "_salt_")
df_small_salting = df_small_salting.repartition(3, "_salt_")

In [ ]:
df_skew_salting.rdd.glom().collect()

In [ ]:
df_small_salting.rdd.glom().collect()


In [ ]:
# apply join()

df_skew_salting.drop("_salt_")
df_small_salting.drop("_salt_")


groups = df_skew_salting.join(df_small_salting, df_skew_salting.OrderID == df_small_salting.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()

## (3) Run a shuffle Join() to see how the skew effects computation resources.¶


In [ ]:
from pyspark.sql.functions import *

# set the number of partitions after shuffling (avoid 200 partitions)
spark.conf.set("spark.sql.shuffle.partitions", 8)

groups = df_skew.join(df_small, df_skew.OrderID == df_small.OrderID, how = "inner")

summary = groups.select(mean(groups.Sales).alias("AVG(Sales)"),
                        stddev(groups.Sales).alias("STD(Sales)"),
                        min(groups.Sales).alias("MIN(Sales)"),
                        max(groups.Sales).alias("MAX(Sales)"),
                       )
summary.show()